# Pipeline de datos: transforms.py + nih.py

Este notebook hace tres cosas:

1. Escribe `src/datasets/transforms.py` en tu Drive.
2. Escribe `src/datasets/nih.py` en tu Drive.
3. Ejecuta smoke tests para verificar que todo funciona con una imagen real.

**Pre-requisitos:**
- Ya corriste `00c_prepare_splits_v4.ipynb` (tienes train.csv/val.csv/test.csv).
- Tienes las 4,999 imagenes NIH en `data/raw/nih/images/`.

**Runtime:** CPU (no necesita GPU todavia).


## 1. Montar Drive e instalar deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR = f'{PROJECT_ROOT}/code'

import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f'CODE_DIR: {CODE_DIR}')


In [ ]:
# torchxrayvision necesita estar instalado en esta sesion
!pip install -q torchxrayvision scipy
print('deps OK')


## 2. Escribir `src/datasets/transforms.py`

In [ ]:
from pathlib import Path

content = r'''"""
src/datasets/transforms.py

Pipeline de preprocesamiento de imagenes para radiografias de torax.

Se usan los transforms nativos de torchxrayvision porque:
  1. XRayCenterCrop ajusta el aspect ratio al centro de la imagen (las CXR
     no siempre vienen cuadradas, y los bordes suelen ser ruido o letras).
  2. XRayResizer escala a 224x224 manteniendo las propiedades del modelo
     preentrenado (los modelos 'res224' de xrv esperan ese tamano).

Ademas se agregan:
  - Normalizacion al rango [-1024, 1024] que es el formato esperado por
    los modelos de xrv (las CXR originales en 16-bit se normalizan asi).
  - Data augmentation opcional (solo en entrenamiento): flips, rotaciones
    pequenas, jitter de brillo. Se mantiene conservador porque en CXR un
    flip horizontal puede alterar la anatomia (el corazon esta a la izquierda
    del paciente, no del observador).

IMPORTANTE sobre flips:
  - El flip horizontal INVIERTE la posicion del corazon. Para clasificacion
    de cardiomegalia esto podria enganar al modelo. Por defecto se DESACTIVA.
  - Se usan solo rotaciones chicas (+-5 grados) y jitter leve de brillo.
"""
from __future__ import annotations

import numpy as np
import torch
import torchvision.transforms as T
import torchxrayvision as xrv


# ---------------------------------------------------------------------------
# Normalizacion
# ---------------------------------------------------------------------------
class NormalizeForXRV:
    """
    Normaliza imagenes 8-bit (0-255) al rango esperado por los modelos xrv:
    [-1024, 1024], float32.

    Formula: xrv.datasets.normalize(img_uint8, maxval=255).

    Las imagenes NIH vienen como PNG 8-bit, pero los modelos xrv fueron
    entrenados con CXR 16-bit. Esta funcion hace el puente.
    """
    def __call__(self, img: np.ndarray) -> np.ndarray:
        # img: np.ndarray uint8 (H, W) o (1, H, W), valores 0-255
        return xrv.datasets.normalize(img, maxval=255, reshape=True)


# ---------------------------------------------------------------------------
# Augmentations (solo para train)
# ---------------------------------------------------------------------------
class RandomRotationSmall:
    """Rotacion aleatoria de +-max_degrees grados."""
    def __init__(self, max_degrees: float = 5.0):
        self.max_degrees = max_degrees

    def __call__(self, img: np.ndarray) -> np.ndarray:
        # img: (1, H, W) float32
        from scipy.ndimage import rotate
        angle = np.random.uniform(-self.max_degrees, self.max_degrees)
        rotated = rotate(img, angle, axes=(1, 2), reshape=False,
                         order=1, mode='nearest')
        return rotated.astype(np.float32)


class RandomBrightnessJitter:
    """Escala la intensidad en [1-factor, 1+factor]."""
    def __init__(self, factor: float = 0.1):
        self.factor = factor

    def __call__(self, img: np.ndarray) -> np.ndarray:
        scale = np.random.uniform(1.0 - self.factor, 1.0 + self.factor)
        return (img * scale).astype(np.float32)


class ToTensor:
    """np.ndarray (1, H, W) -> torch.Tensor."""
    def __call__(self, img: np.ndarray) -> torch.Tensor:
        return torch.from_numpy(img).float()


# ---------------------------------------------------------------------------
# Pipelines finales
# ---------------------------------------------------------------------------
def build_transform(image_size: int = 224, augment: bool = False
                    ) -> T.Compose:
    """
    Construye el pipeline de preprocesamiento.

    Args:
        image_size: resolucion objetivo (224 para modelos *-res224).
        augment: si True, agrega augmentations ligeras (solo para train).

    Returns:
        torchvision.transforms.Compose aplicable a una imagen numpy (1, H, W).
    """
    steps = [
        # 1. Normalizar a rango xrv (mantiene shape (1, H, W))
        NormalizeForXRV(),
        # 2. Center crop + resize (xrv spera input (1, H, W))
        xrv.datasets.XRayCenterCrop(),
        xrv.datasets.XRayResizer(image_size),
    ]

    if augment:
        steps.extend([
            RandomRotationSmall(max_degrees=5.0),
            RandomBrightnessJitter(factor=0.1),
        ])

    # 3. Convertir a tensor PyTorch
    steps.append(ToTensor())

    return T.Compose(steps)


def build_train_transform(image_size: int = 224) -> T.Compose:
    """Pipeline para el split de train (con augmentation)."""
    return build_transform(image_size=image_size, augment=True)


def build_eval_transform(image_size: int = 224) -> T.Compose:
    """Pipeline para val/test (sin augmentation)."""
    return build_transform(image_size=image_size, augment=False)


# ---------------------------------------------------------------------------
# Smoke test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # Imagen sintetica 8-bit (1024x1024, como vienen las NIH)
    fake_img = np.random.randint(0, 256, size=(1024, 1024), dtype=np.uint8)

    print("Test eval transform:")
    tfm_eval = build_eval_transform(image_size=224)
    out = tfm_eval(fake_img)
    print(f"  shape: {out.shape}, dtype: {out.dtype}")
    print(f"  rango: [{out.min():.2f}, {out.max():.2f}]")

    print("\nTest train transform (con augmentation):")
    tfm_train = build_train_transform(image_size=224)
    out = tfm_train(fake_img)
    print(f"  shape: {out.shape}, dtype: {out.dtype}")
    print(f"  rango: [{out.min():.2f}, {out.max():.2f}]")

    print("\nOK: transforms.py funciona correctamente.")
'''
path = f'{CODE_DIR}/src/datasets/transforms.py'
Path(path).parent.mkdir(parents=True, exist_ok=True)
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


## 3. Escribir `src/datasets/nih.py`

In [ ]:
content = r'''"""
src/datasets/nih.py

Dataset de PyTorch para el baseline binario NIH (No Finding vs Cardiomegaly).

Lee los CSVs generados por scripts/prepare_splits.py y devuelve:
  (image_tensor, label) por cada muestra.

Columnas esperadas en los CSVs:
  - Image Index (nombre del archivo .png)
  - Finding Labels (texto original, por si se necesita)
  - Patient ID
  - View Position
  - label (0 o 1)

Las imagenes se cargan desde {paths.data_raw}/images/ en escala de grises.
Se aplican los transforms definidos en src/datasets/transforms.py.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset


class NIHDataset(Dataset):
    """
    Dataset NIH para clasificacion binaria.

    Args:
        csv_path: ruta al CSV generado por prepare_splits (train/val/test).
        images_dir: carpeta que contiene los archivos .png del NIH.
        transform: callable a aplicar sobre cada imagen (de transforms.py).

    Cada __getitem__ retorna:
        {
          'image': torch.Tensor (1, H, W), float32 en rango [-1024, 1024],
          'label': torch.Tensor escalar, long (0 o 1),
          'image_index': str (nombre del archivo, util para debugging),
        }
    """

    def __init__(self, csv_path: str | Path, images_dir: str | Path,
                 transform=None):
        self.csv_path = Path(csv_path)
        self.images_dir = Path(images_dir)
        self.transform = transform

        if not self.csv_path.exists():
            raise FileNotFoundError(f"CSV no encontrado: {self.csv_path}")
        if not self.images_dir.exists():
            raise FileNotFoundError(f"images_dir no encontrado: {self.images_dir}")

        self.df = pd.read_csv(self.csv_path)
        self._validate_columns()

    def _validate_columns(self) -> None:
        required = {'Image Index', 'label', 'Patient ID'}
        missing = required - set(self.df.columns)
        if missing:
            raise ValueError(f"Columnas faltantes en {self.csv_path}: {missing}")

    # ------------------------------------------------------------------
    # Protocol
    # ------------------------------------------------------------------
    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        image_name = row['Image Index']
        label = int(row['label'])

        image = self._load_image(image_name)

        if self.transform is not None:
            image = self.transform(image)

        return {
            'image': image,
            'label': torch.tensor(label, dtype=torch.long),
            'image_index': image_name,
        }

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    def _load_image(self, image_name: str) -> np.ndarray:
        """
        Carga una imagen como array (1, H, W) uint8 en escala de grises.
        Las CXR del NIH estan en escala de grises pero algunas vienen en RGB
        (duplicando canales). Forzamos modo L para consistencia.
        """
        img_path = self.images_dir / image_name
        if not img_path.exists():
            raise FileNotFoundError(f"Imagen no encontrada: {img_path}")

        with Image.open(img_path) as pil_img:
            pil_img = pil_img.convert('L')  # forzar gris
            arr = np.array(pil_img, dtype=np.uint8)

        # (H, W) -> (1, H, W) para compatibilidad con transforms xrv
        return arr[np.newaxis, :, :]

    # ------------------------------------------------------------------
    # Conveniencia
    # ------------------------------------------------------------------
    def class_distribution(self) -> dict:
        """Distribucion de clases en el split."""
        counts = self.df['label'].value_counts().to_dict()
        return {
            'positive': int(counts.get(1, 0)),
            'negative': int(counts.get(0, 0)),
            'total': len(self.df),
        }


# ---------------------------------------------------------------------------
# Factory functions
# ---------------------------------------------------------------------------
def build_nih_datasets(cfg, train_transform, eval_transform
                       ) -> tuple[NIHDataset, NIHDataset, NIHDataset]:
    """
    Crea los tres datasets (train, val, test) usando la config del proyecto.

    Args:
        cfg: objeto Config cargado con load_config().
        train_transform: transform para train (con augmentation).
        eval_transform: transform para val y test (sin augmentation).

    Returns:
        (train_ds, val_ds, test_ds)
    """
    processed = Path(cfg.paths['data_processed'])
    images_dir = Path(cfg.paths['data_raw']) / cfg.dataset['images_subdir']

    train_ds = NIHDataset(
        csv_path=processed / "train.csv",
        images_dir=images_dir,
        transform=train_transform,
    )
    val_ds = NIHDataset(
        csv_path=processed / "val.csv",
        images_dir=images_dir,
        transform=eval_transform,
    )
    test_ds = NIHDataset(
        csv_path=processed / "test.csv",
        images_dir=images_dir,
        transform=eval_transform,
    )
    return train_ds, val_ds, test_ds


# ---------------------------------------------------------------------------
# Smoke test
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    import sys
    # Permitir imports desde el root del proyecto
    code_dir = Path(__file__).resolve().parent.parent.parent
    if str(code_dir) not in sys.path:
        sys.path.insert(0, str(code_dir))

    from src.training.utils import load_config
    from src.datasets.transforms import build_train_transform, build_eval_transform

    cfg = load_config(code_dir / "configs" / "baseline.yaml")

    train_tfm = build_train_transform(image_size=cfg.preprocessing['image_size'])
    eval_tfm = build_eval_transform(image_size=cfg.preprocessing['image_size'])

    train_ds, val_ds, test_ds = build_nih_datasets(cfg, train_tfm, eval_tfm)

    print(f"Train: {len(train_ds)} muestras | dist: {train_ds.class_distribution()}")
    print(f"Val:   {len(val_ds)} muestras | dist: {val_ds.class_distribution()}")
    print(f"Test:  {len(test_ds)} muestras | dist: {test_ds.class_distribution()}")

    # Cargar una muestra
    sample = train_ds[0]
    print(f"\nMuestra [0]:")
    print(f"  image shape: {sample['image'].shape}")
    print(f"  image dtype: {sample['image'].dtype}")
    print(f"  image range: [{sample['image'].min():.2f}, {sample['image'].max():.2f}]")
    print(f"  label: {sample['label'].item()}")
    print(f"  image_index: {sample['image_index']}")
'''
path = f'{CODE_DIR}/src/datasets/nih.py'
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


## 4. Smoke test: transforms.py con imagen sintetica

In [ ]:
# Limpiar cache de imports por si ya se habia cargado
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'):
        del sys.modules[mod]

import numpy as np
from src.datasets.transforms import build_train_transform, build_eval_transform

# Imagen sintetica estilo NIH (1024x1024, 8-bit gris)
fake_img = np.random.randint(0, 256, size=(1024, 1024), dtype=np.uint8)

print('Test eval transform:')
tfm_eval = build_eval_transform(image_size=224)
out = tfm_eval(fake_img)
print(f'  shape: {out.shape}')
print(f'  dtype: {out.dtype}')
print(f'  rango: [{out.min():.2f}, {out.max():.2f}]')

print('\nTest train transform (con augmentation):')
tfm_train = build_train_transform(image_size=224)
out = tfm_train(fake_img)
print(f'  shape: {out.shape}')
print(f'  dtype: {out.dtype}')
print(f'  rango: [{out.min():.2f}, {out.max():.2f}]')


## 5. Smoke test: nih.py con datos reales

Carga los 3 datasets desde los CSVs del split y verifica que puede leer
una imagen real del Drive.

In [ ]:
from src.training.utils import load_config
from src.datasets.nih import build_nih_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform

cfg = load_config(f'{CODE_DIR}/configs/baseline.yaml')

train_tfm = build_train_transform(image_size=cfg.preprocessing['image_size'])
eval_tfm = build_eval_transform(image_size=cfg.preprocessing['image_size'])

train_ds, val_ds, test_ds = build_nih_datasets(cfg, train_tfm, eval_tfm)

print(f'Train: {len(train_ds)} muestras | dist: {train_ds.class_distribution()}')
print(f'Val:   {len(val_ds)} muestras | dist: {val_ds.class_distribution()}')
print(f'Test:  {len(test_ds)} muestras | dist: {test_ds.class_distribution()}')


## 6. Cargar una muestra real y visualizarla

In [ ]:
import matplotlib.pyplot as plt

# Cargar la primera muestra
sample = train_ds[0]
print(f"image shape: {sample['image'].shape}")
print(f"image dtype: {sample['image'].dtype}")
print(f"image range: [{sample['image'].min():.2f}, {sample['image'].max():.2f}]")
print(f"label: {sample['label'].item()} ({'Cardiomegaly' if sample['label'].item() == 1 else 'No Finding'})")
print(f"image_index: {sample['image_index']}")

# Visualizar
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
img = sample['image'].squeeze().numpy()
ax.imshow(img, cmap='gray')
ax.set_title(f"{sample['image_index']} | label={sample['label'].item()}")
ax.axis('off')
plt.tight_layout()
plt.show()


## 7. Probar un batch con DataLoader

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    train_ds,
    batch_size=cfg.training['batch_size'],
    shuffle=True,
    num_workers=0,  # en Colab a veces num_workers>0 da problemas, usar 0 primero
)

batch = next(iter(loader))
print(f"Batch cargado:")
print(f"  images shape: {batch['image'].shape}")
print(f"  labels shape: {batch['label'].shape}")
print(f"  labels en el batch: {batch['label'].tolist()}")
print(f"  positivos: {(batch['label'] == 1).sum().item()} / {len(batch['label'])}")


## 8. Visualizar un grid de 8 imagenes del batch

In [ ]:
import matplotlib.pyplot as plt

n_show = min(8, len(batch['image']))
fig, axes = plt.subplots(2, 4, figsize=(14, 8))
for i, ax in enumerate(axes.flat):
    if i >= n_show:
        ax.axis('off')
        continue
    img = batch['image'][i].squeeze().numpy()
    label = batch['label'][i].item()
    ax.imshow(img, cmap='gray')
    color = 'red' if label == 1 else 'green'
    ax.set_title(f'label={label} ({"Cardio" if label else "Normal"})',
                 color=color)
    ax.axis('off')
plt.suptitle('Batch de entrenamiento (despues de preprocesamiento + augmentation)')
plt.tight_layout()
plt.show()


## 9. Siguiente paso

Si todo lo anterior funcionó:
- ✅ Los transforms preprocesan las imagenes al formato esperado por xrv.
- ✅ El Dataset carga correctamente las imagenes desde Drive.
- ✅ El DataLoader arma batches sin errores.

**El pipeline de datos esta COMPLETO.**

Vuelve al chat para recibir:
1. `src/models/baseline.py` - DenseNet121 de torchxrayvision adaptado para 2 clases.
2. `src/evaluation/metrics.py` - AUC, accuracy, precision, sensibilidad, especificidad.

Con esos dos archivos estaremos listos para el loop de entrenamiento.


In [ ]:
import numpy as np
import torchxrayvision as xrv
from PIL import Image

# Cargar una imagen real
img_path = f'{PROJECT_ROOT}/data/raw/nih/images/00000001_002.png'
pil_img = Image.open(img_path).convert('L')
arr = np.array(pil_img, dtype=np.uint8)
print(f'1. Original uint8: shape={arr.shape}, range=[{arr.min()}, {arr.max()}]')

# Paso 1: normalize
normalized = xrv.datasets.normalize(arr, maxval=255, reshape=True)
print(f'2. Tras normalize: shape={normalized.shape}, range=[{normalized.min():.2f}, {normalized.max():.2f}]')

# Paso 2: center crop
crop = xrv.datasets.XRayCenterCrop()
cropped = crop(normalized)
print(f'3. Tras crop:      shape={cropped.shape}, range=[{cropped.min():.2f}, {cropped.max():.2f}]')

# Paso 3: resize
resize = xrv.datasets.XRayResizer(224)
resized = resize(cropped)
print(f'4. Tras resize:    shape={resized.shape}, range=[{resized.min():.2f}, {resized.max():.2f}]')